# 6. Interpretation: per-position attribution, and assembling k-mers into motifs

Two steps. First, ask which input positions drive a prediction, using input times
gradient. Second, turn those per-position scores into something readable, which is what
`metamotif` does: it finds subsequences whose scores are higher than chance and assembles
them into a consensus.

**Read the next section before using `model.explain()`.** It does not attribute what most
people assume it attributes.

In [ ]:
import torch

torch.set_num_threads(2)

import numpy as np
import parnet_tutorials as pt
from parnet.models import RBPNet
from parnet.utils import sequence_to_onehot

MODEL = 'parnet-21m-penalty5'  # change this one line to run the notebook on another model
model = RBPNet.from_pretrained(pt.model_path(MODEL))
task_names = pt.task_names('encode-223')

## What `model.explain()` actually attributes

The library's `explain()` builds this scalar and attributes it with Captum's
`InputXGradient`:

```python
pred = self.forward(inputs)[track][:, task_idx, :].softmax(dim=-1)
return (pred * pred.detach()).sum(dim=-1)
```

`pred.detach()` means gradient flows through the first factor only, so the attributed
scalar behaves as one half of `sum_j p_j^2`. That quantity measures how **concentrated**
the predicted profile is, not how large it is at any particular position.

So `explain()` answers "which input positions make this profile peaked?" It does not
answer "which input positions raise predicted signal at position j?", which is what a
per-position attribution usually means, and it is not specific to any region of the
window.

That distinction is recorded as an open discrepancy against the Methods description in
the refactor handover (section 32) and has not been resolved. **Do not present
`explain()` output as the paper's attribution without checking that first.** If you want
a region-specific attribution, state the target scalar yourself, as below.

In [ ]:
rng = np.random.default_rng(0)
motif = 'TGCATG'                      # RBFOX-like
sequence = list(rng.choice(list('ACGT'), size=600))
for start in (180, 300, 420):         # plant it three times
    sequence[start:start + len(motif)] = list(motif)
sequence = ''.join(sequence)

# Which tracks exist depends on the model: the 21M emits 223 tracks across both cell
# types, each 3M backbone only its split's training RBPs, all HepG2. Ask for a role.
rbp = pt.demo_track(MODEL)
task = pt.track_index(MODEL, rbp)
x = sequence_to_onehot(sequence).float().unsqueeze(0)
print(f'{rbp} is task {task}; motif planted at 180, 300, 420')

### State the target scalar explicitly

Here the target is the predicted log-density summed over a window of interest. That is a
quantity with a clear meaning: "how much signal does the model put in this region?".

In [ ]:
def attribute(sequence_onehot, task_idx, region=None, track='target'):
    """Input x gradient of the predicted log-density summed over `region`.

    region=None means the whole window. Returns (4, L) attribution, one value per base
    per position, and the per-position total used for plotting and for metamotif.
    """
    inputs = sequence_onehot.clone().requires_grad_(True)
    logprob = model(inputs)[track][0, task_idx]        # (L,)
    target = logprob.sum() if region is None else logprob[region].sum()
    (gradient,) = torch.autograd.grad(target, inputs)
    attribution = (gradient * inputs).detach()[0]       # (4, L)
    return attribution, attribution.sum(0).numpy()


# Attribute the region around the central planted motif.
region = slice(280, 320)
attribution, per_position = attribute(x, task, region=region)
print('attribution:', tuple(attribution.shape))

for start in (180, 300, 420):
    window = per_position[start:start + len(motif)]
    print(f'  planted motif at {start}: mean score {window.mean():+.4f}')

background = np.delete(per_position, np.r_[180:186, 300:306, 420:426])
print(f'  background                : mean score {background.mean():+.4f}')

The region-specific target picks out the motif copy inside the region much more strongly
than the copies outside it.

The decisive comparison is not which method scores the central motif higher. It is
whether the attribution **moves when you ask about a different region**. The explicit
target takes a region argument; `model.explain()` has none, because the scalar it
attributes is a property of the whole profile.

In [ ]:
import pandas as pd
pd.set_option('display.width', 200)

library_per_position = model.explain(x, task_idx=task, track='target')[0].detach().sum(0).numpy()

# Ask about each planted motif in turn, and see which motif the attribution favours.
rows = []
for centre in (180, 300, 420):
    _, scores = attribute(x, task, region=slice(centre - 20, centre + 26))
    row = {'target region': f'{centre - 20}:{centre + 26}'}
    for planted in (180, 300, 420):
        row[f'motif@{planted}'] = f'{scores[planted:planted + 6].mean():+8.3f}'
    row['argmax motif'] = max((180, 300, 420),
                              key=lambda p: scores[p:p + 6].mean())
    rows.append(row)

row = {'target region': 'model.explain() (none)'}
for planted in (180, 300, 420):
    row[f'motif@{planted}'] = f'{library_per_position[planted:planted + 6].mean():+8.4f}'
row['argmax motif'] = max((180, 300, 420),
                          key=lambda p: library_per_position[p:p + 6].mean())
rows.append(row)

print(pd.DataFrame(rows).to_string(index=False))
print()
print('The explicit target follows the region it was asked about.')
print('model.explain() returns one answer, because its scalar has no region in it.')
print('Both are legitimate. They answer different questions.')

## Plot the attribution

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(per_position, linewidth=0.8, color='#333333')
for start in (180, 300, 420):
    ax.axvspan(start, start + len(motif), color='tab:orange', alpha=0.35)
ax.axvspan(region.start, region.stop, color='tab:blue', alpha=0.10)
ax.set_xlabel('position (nt)')
ax.set_ylabel('input x gradient')
ax.set_title(f'{rbp}: orange = planted {motif}, blue = attributed region', fontsize=10)
fig.tight_layout()

results = pt.project_root() / 'results'
results.mkdir(exist_ok=True)
fig.savefig(results / '06_attribution.png', dpi=120)
print('saved', results / '06_attribution.png')

## From scores to motifs with metamotif

`metamotif.search.search()` takes the per-position scores for one sequence and returns
subsequences whose summed score is higher than a null built by shuffling. It grows a seed
outward while the result stays significant, so the length is found rather than fixed.

This is the step that turns "position 301 matters" into "UGCAUG matters".

In [ ]:
try:
    from metamotif.search import search as metamotif_search
    have_metamotif = True
except ImportError:
    have_metamotif = False
    print('metamotif is not installed. Install it with:')
    print('    pip install "parnet-tutorials[motifs]"')
    print('or  pip install git+https://github.com/project-parnet/metamotif.git')
    print()
    print('Falling back to a plain top-scoring-window scan so the notebook still runs.')

scores = per_position.astype(float)

if have_metamotif:
    found = metamotif_search(scores, sig_p=0.01, seed_size=2, max_size=12)
    print(f'metamotif returned {len(found)} subsequences')
    hits = []
    for entry in found[:10]:
        start, end = entry[0], entry[1]
        hits.append({'start': start, 'end': end,
                     'sequence': sequence[start:end],
                     'score': float(scores[start:end].sum())})
else:
    # Equivalent idea, far cruder: best-scoring fixed-width windows, non-overlapping.
    width = len(motif)
    running = np.convolve(scores, np.ones(width), mode='valid')
    hits, taken = [], np.zeros(len(running), dtype=bool)
    for position in np.argsort(-running):
        if taken[max(0, position - width):position + width].any():
            continue
        taken[position] = True
        hits.append({'start': int(position), 'end': int(position + width),
                     'sequence': sequence[position:position + width],
                     'score': float(running[position])})
        if len(hits) == 10:
            break

print(pd.DataFrame(hits).to_string(index=False))

The planted motif should appear among the top hits. On real data you would run this
across many peak-centred windows for one RBP and count which subsequences recur, which is
what makes the result a motif rather than an observation about one sequence.

## What to take away

1. An attribution is only as meaningful as the scalar it attributes. State that scalar.
2. `model.explain()` attributes profile concentration, not region-specific signal, and
   that differs from the Methods description. Check before citing it.
3. Input times gradient is the paper's method. Captum offers alternatives, unranked.
4. `metamotif` turns per-position scores into consensus subsequences, and is the step
   that makes attribution readable.